# Notebook 45
# Phase 2 – LSTM Baseline

## Objective

Investigate whether an LSTM trained on raw multivariate sensor sequences can outperform the classical XGBoost baseline based on handcrafted features.

---

## Research Question (RQ1)

Can an LSTM learn temporal driving behaviour directly from synchronized raw sensor signals?

---

## Baseline

Model:
- XGBoost

Window Size:
- 120

Threshold:
- 0.30

Evaluation:
- Trip-Based GroupKFold

---

## LSTM Design Decisions

Sequence Length : 120

Stride : 10

Input : Raw Sensor Signals

Evaluation : Same protocol as Phase 1

Target : Behaviour Classification

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler

print(torch.__version__)

2.11.0+cpu


In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
PROJECT_PATH = Path("/content/drive/MyDrive/UAH_Project")

SRC_PATH = PROJECT_PATH / "src"
DATASET_PATH = PROJECT_PATH / "datasets" / "raw" / "UAH-DRIVESET-v1"
RESULTS_PATH = PROJECT_PATH / "results"

sys.path.append(str(SRC_PATH))

print(PROJECT_PATH)

/content/drive/MyDrive/UAH_Project


In [10]:
SEQUENCE_LENGTH = 120
STRIDE = 10
BATCH_SIZE = 32
LEARNING_RATE = 1e-3

print(f"Sequence Length : {SEQUENCE_LENGTH}")
print(f"Stride          : {STRIDE}")
print(f"Batch Size      : {BATCH_SIZE}")

Sequence Length : 120
Stride          : 10
Batch Size      : 32


In [11]:
from preprocessor import (
    load_accelerometer,
    load_gps,
    load_lane_detection,
    load_vehicle_detection,
)

print("Preprocessor imported successfully.")

Preprocessor imported successfully.


In [29]:
import preprocessor

print(preprocessor.__file__)

print(dir(preprocessor))

/content/drive/MyDrive/UAH_Project/src/preprocessor.py
['ACC_COLUMNS', 'GPS_COLUMNS', 'LANE_COLUMNS', 'Path', 'VEHICLE_COLUMNS', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'get_trip_files', 'load_accelerometer', 'load_gps', 'load_lane_detection', 'load_trip', 'load_vehicle_detection', 'merge_trip', 'pd']


In [13]:
from pathlib import Path

preprocessor_path = PROJECT_PATH / "src" / "preprocessor.py"

print(preprocessor_path.read_text()[-300:])

ensor file paths for a trip.
    """

    return {
        "accelerometer": trip_path / "RAW_ACCELEROMETERS.txt",
        "gps": trip_path / "RAW_GPS.txt",
        "lane": trip_path / "PROC_LANE_DETECTION.txt",
        "vehicle": trip_path / "PROC_VEHICLE_DETECTION.txt",
    }
    # TEST_ALEYNA_1234


In [14]:
from preprocessor import get_trip_files

trip_path = (
    DATASET_PATH
    / "D1"
    / "20151111123124-25km-D1-NORMAL-MOTORWAY"
)

files = get_trip_files(trip_path)

for name, path in files.items():
    print(f"{name:15} -> {path}")

accelerometer   -> /content/drive/MyDrive/UAH_Project/datasets/raw/UAH-DRIVESET-v1/D1/20151111123124-25km-D1-NORMAL-MOTORWAY/RAW_ACCELEROMETERS.txt
gps             -> /content/drive/MyDrive/UAH_Project/datasets/raw/UAH-DRIVESET-v1/D1/20151111123124-25km-D1-NORMAL-MOTORWAY/RAW_GPS.txt
lane            -> /content/drive/MyDrive/UAH_Project/datasets/raw/UAH-DRIVESET-v1/D1/20151111123124-25km-D1-NORMAL-MOTORWAY/PROC_LANE_DETECTION.txt
vehicle         -> /content/drive/MyDrive/UAH_Project/datasets/raw/UAH-DRIVESET-v1/D1/20151111123124-25km-D1-NORMAL-MOTORWAY/PROC_VEHICLE_DETECTION.txt


In [26]:
import importlib
import preprocessor

importlib.reload(preprocessor)

<module 'preprocessor' from '/content/drive/MyDrive/UAH_Project/src/preprocessor.py'>

In [19]:
from preprocessor import load_trip

trip = load_trip(trip_path)

for sensor_name, df in trip.items():
    print("=" * 60)
    print(sensor_name.upper())
    print(df.shape)
    display(df.head())

ACCELEROMETER
(8668, 11)


,timestamp,active,acc_x,acc_y,acc_z,acc_x_kf,acc_y_kf,acc_z_kf,roll,pitch,yaw
0,0.69,0,0.066,0.12,0.033,0.012,0.022,0.006,-1.487,-0.223,-0.373
1,0.69,0,0.083,0.15,0.049,0.049,0.088,0.028,-1.652,-0.248,-0.415
2,0.70,0,0.083,0.15,0.049,0.070,0.126,0.041,-1.652,-0.248,-0.415
3,0.70,0,0.083,0.15,0.049,0.078,0.141,0.046,-1.652,-0.248,-0.415
4,0.70,0,0.083,0.15,0.049,0.081,0.147,0.048,-1.652,-0.248,-0.415


GPS
(864, 12)


,timestamp,speed,latitude,longitude,altitude,gps_quality,satellites,heading,extra_1,extra_2,extra_3,extra_4
0,11.88,0.0,40.505939,-3.360690,610.3,30,201,-1.0,-1.000,-9,-9,0
1,12.90,78.8,40.506252,-3.355367,636.5,16,30,272.5,87.539,0,0,0
2,13.93,82.7,40.505802,-3.355622,605.9,4,5,273.9,1.406,0,0,0
3,14.89,85.5,40.505798,-3.355894,606.3,6,5,274.6,2.109,0,0,0
4,15.88,85.7,40.505810,-3.356182,606.9,6,5,274.9,1.055,0,0,0


LANE
(20657, 5)


,time,lane_offset,phi,road_width,lane_state
0,10.89,-9.0,-9.0,-9.0,-9
1,10.94,-9.0,-9.0,-9.0,-9
2,11.02,-9.0,-9.0,-9.0,-9
3,11.09,-9.0,-9.0,-9.0,-9
4,11.15,-9.0,-9.0,-9.0,-9


VEHICLE
(4599, 5)


,time,front_distance,relative_speed,vehicle_state,confidence
0,12.58,-1.0,-1.0,0,76.8
1,12.70,-1.0,-1.0,0,76.8
2,12.80,-1.0,-1.0,0,78.8
3,12.97,-1.0,-1.0,0,78.8
4,13.06,-1.0,-1.0,0,78.8


In [22]:
from preprocessor import merge_trip

merged_df = merge_trip(trip_path)

print(merged_df.shape)

display(merged_df.head())

(8668, 32)


,timestamp,active,acc_x,acc_y,acc_z,acc_x_kf,acc_y_kf,acc_z_kf,roll,pitch,...,time_x,lane_offset,phi,road_width,lane_state,time_y,front_distance,relative_speed,vehicle_state,confidence
0,0.69,0,0.066,0.12,0.033,0.012,0.022,0.006,-1.487,-0.223,...,10.89,-9.0,-9.0,-9.0,-9,12.58,-1.0,-1.0,0,76.8
1,0.69,0,0.083,0.15,0.049,0.049,0.088,0.028,-1.652,-0.248,...,10.89,-9.0,-9.0,-9.0,-9,12.58,-1.0,-1.0,0,76.8
2,0.70,0,0.083,0.15,0.049,0.078,0.141,0.046,-1.652,-0.248,...,10.89,-9.0,-9.0,-9.0,-9,12.58,-1.0,-1.0,0,76.8
3,0.70,0,0.083,0.15,0.049,0.070,0.126,0.041,-1.652,-0.248,...,10.89,-9.0,-9.0,-9.0,-9,12.58,-1.0,-1.0,0,76.8
4,0.70,0,0.083,0.15,0.049,0.082,0.149,0.048,-1.652,-0.248,...,10.89,-9.0,-9.0,-9.0,-9,12.58,-1.0,-1.0,0,76.8


In [23]:
print("=" * 80)
print("Merged DataFrame Info")
print("=" * 80)

merged_df.info()

Merged DataFrame Info
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8668 entries, 0 to 8667
Data columns (total 32 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   timestamp       8668 non-null   float64
 1   active          8668 non-null   int64  
 2   acc_x           8668 non-null   float64
 3   acc_y           8668 non-null   float64
 4   acc_z           8668 non-null   float64
 5   acc_x_kf        8668 non-null   float64
 6   acc_y_kf        8668 non-null   float64
 7   acc_z_kf        8668 non-null   float64
 8   roll            8668 non-null   float64
 9   pitch           8668 non-null   float64
 10  yaw             8668 non-null   float64
 11  speed           8668 non-null   float64
 12  latitude        8668 non-null   float64
 13  longitude       8668 non-null   float64
 14  altitude        8668 non-null   float64
 15  gps_quality     8668 non-null   int64  
 16  satellites      8668 non-null   int64  
 17  heading    

In [24]:
print("=" * 80)
print("Missing Values")
print("=" * 80)

merged_df.isna().sum().sort_values(ascending=False)

Missing Values


,0
timestamp,0
active,0
acc_x,0
acc_y,0
acc_z,0
acc_x_kf,0
acc_y_kf,0
acc_z_kf,0
roll,0
pitch,0


In [35]:
import importlib
import preprocessor

importlib.reload(preprocessor)

<module 'preprocessor' from '/content/drive/MyDrive/UAH_Project/src/preprocessor.py'>

In [32]:
from preprocessor import clean_trip

clean_df = clean_trip(merged_df)

print(clean_df.shape)

display(clean_df.head())

(8668, 13)


,acc_x,acc_y,acc_z,roll,pitch,yaw,speed,heading,lane_offset,phi,front_distance,relative_speed,vehicle_state
0,0.066,0.12,0.033,-1.487,-0.223,-0.373,0.0,-1.0,-9.0,-9.0,-1.0,-1.0,0
1,0.083,0.15,0.049,-1.652,-0.248,-0.415,0.0,-1.0,-9.0,-9.0,-1.0,-1.0,0
2,0.083,0.15,0.049,-1.652,-0.248,-0.415,0.0,-1.0,-9.0,-9.0,-1.0,-1.0,0
3,0.083,0.15,0.049,-1.652,-0.248,-0.415,0.0,-1.0,-9.0,-9.0,-1.0,-1.0,0
4,0.083,0.15,0.049,-1.652,-0.248,-0.415,0.0,-1.0,-9.0,-9.0,-1.0,-1.0,0


In [50]:
from data_loader import create_sequences

X = clean_df.to_numpy()

X_seq = create_sequences(X)

print("Original Shape :", X.shape)
print("Sequence Shape :", X_seq.shape)

Original Shape : (8668, 13)
Sequence Shape : (855, 120, 13)


In [51]:
import importlib
import data_loader

importlib.reload(data_loader)

print(dir(data_loader))

['DataLoader', 'Dataset', 'FEATURE_COLUMNS', 'LABEL_MAPPING', 'Path', 'SEQUENCE_LENGTH', 'STRIDE', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'build_dataset', 'clean_trip', 'create_sequences', 'extract_trip_info', 'get_all_trip_paths', 'merge_trip', 'np', 'pd', 'torch']


In [52]:
from data_loader import extract_trip_info

info = extract_trip_info(
    "20151111123124-25km-D1-NORMAL-MOTORWAY"
)

print(info)

{'driver': 'D1', 'behaviour': 'NORMAL', 'road': 'MOTORWAY', 'label': 0}


In [53]:
from data_loader import get_all_trip_paths

trip_paths = get_all_trip_paths(DATASET_PATH)

print(f"Toplam Trip Sayısı : {len(trip_paths)}")

print()

for trip in trip_paths[:5]:
    print(trip.name)

Toplam Trip Sayısı : 41

20151110175712-16km-D1-NORMAL1-SECONDARY
20151110180824-16km-D1-NORMAL2-SECONDARY
20151111123124-25km-D1-NORMAL-MOTORWAY
20151111125233-24km-D1-AGGRESSIVE-MOTORWAY
20151111132348-25km-D1-DROWSY-MOTORWAY


In [55]:
import importlib
import data_loader

importlib.reload(data_loader)

<module 'data_loader' from '/content/drive/MyDrive/UAH_Project/src/data_loader.py'>

In [57]:
trip_paths = get_all_trip_paths(DATASET_PATH)

for trip in trip_paths:
    print(trip.name)

20151110175712-16km-D1-NORMAL1-SECONDARY
20151110180824-16km-D1-NORMAL2-SECONDARY
20151111123124-25km-D1-NORMAL-MOTORWAY
20151111125233-24km-D1-AGGRESSIVE-MOTORWAY
20151111132348-25km-D1-DROWSY-MOTORWAY
20151111134545-16km-D1-AGGRESSIVE-SECONDARY
20151111135612-13km-D1-DROWSY-SECONDARY
20151120131714-26km-D2-NORMAL-MOTORWAY
20151120133502-26km-D2-AGGRESSIVE-MOTORWAY
20151120135152-25km-D2-DROWSY-MOTORWAY
20151120160904-16km-D2-NORMAL1-SECONDARY
20151120162105-17km-D2-NORMAL2-SECONDARY
20151120163350-16km-D2-AGGRESSIVE-SECONDARY
20151120164606-16km-D2-DROWSY-SECONDARY
20151126110502-26km-D3-NORMAL-MOTORWAY
20151126113754-26km-D3-DROWSY-MOTORWAY
20151126124208-16km-D3-NORMAL1-SECONDARY
20151126125458-16km-D3-NORMAL2-SECONDARY
20151126130707-16km-D3-AGGRESSIVE-SECONDARY
20151126132013-17km-D3-DROWSY-SECONDARY
20151126134736-26km-D3-AGGRESSIVE-MOTORWAY
20151203171800-16km-D4-NORMAL1-SECONDARY
20151203173103-17km-D4-NORMAL2-SECONDARY
20151203174324-16km-D4-AGGRESSIVE-SECONDARY
2015120317563

In [58]:
trip_paths = get_all_trip_paths(DATASET_PATH)

for trip in trip_paths:
    try:
        info = extract_trip_info(trip.name)
    except Exception:
        print("Problemli klasör:", trip.name)

Problemli klasör: 20151110175712-16km-D1-NORMAL1-SECONDARY
Problemli klasör: 20151110180824-16km-D1-NORMAL2-SECONDARY
Problemli klasör: 20151120160904-16km-D2-NORMAL1-SECONDARY
Problemli klasör: 20151120162105-17km-D2-NORMAL2-SECONDARY
Problemli klasör: 20151126124208-16km-D3-NORMAL1-SECONDARY
Problemli klasör: 20151126125458-16km-D3-NORMAL2-SECONDARY
Problemli klasör: 20151203171800-16km-D4-NORMAL1-SECONDARY
Problemli klasör: 20151203173103-17km-D4-NORMAL2-SECONDARY
Problemli klasör: 20151211162829-16km-D5-NORMAL1-SECONDARY
Problemli klasör: 20151211164124-17km-D5-NORMAL2-SECONDARY
Problemli klasör: icons


In [59]:
trip = trip_paths[0]

print(trip.name)
print(trip.name.split("-"))
print(len(trip.name.split("-")))

20151110175712-16km-D1-NORMAL1-SECONDARY
['20151110175712', '16km', 'D1', 'NORMAL1', 'SECONDARY']
5


In [60]:
import importlib
import data_loader

importlib.reload(data_loader)

from data_loader import build_dataset

In [61]:
from data_loader import build_dataset

X, y, groups = build_dataset(DATASET_PATH)

print("X :", X.shape)
print("y :", y.shape)
print("groups :", groups.shape)

X : (30676, 120, 13)
y : (30676,)
groups : (30676,)


In [62]:
import numpy as np

print("X Shape :", X.shape)
print("y Shape :", y.shape)

print()

print("Class Distribution")
print(np.unique(y, return_counts=True))

X Shape : (30676, 120, 13)
y Shape : (30676,)

Class Distribution
(array([0, 1, 2]), array([12991,  9846,  7839]))


In [63]:
print("Trip Count :", len(np.unique(groups)))

Trip Count : 40


In [65]:
trip_paths = get_all_trip_paths(DATASET_PATH)

print("Trip Sayısı:", len(trip_paths))

print("\nSon 5 klasör:")

for trip in trip_paths[-5:]:
    print(trip)

Trip Sayısı: 41

Son 5 klasör:
/content/drive/MyDrive/UAH_Project/datasets/raw/UAH-DRIVESET-v1/D6/20151217164730-25km-D6-DROWSY-MOTORWAY
/content/drive/MyDrive/UAH_Project/datasets/raw/UAH-DRIVESET-v1/D6/20151221112434-17km-D6-NORMAL-SECONDARY
/content/drive/MyDrive/UAH_Project/datasets/raw/UAH-DRIVESET-v1/D6/20151221113846-16km-D6-DROWSY-SECONDARY
/content/drive/MyDrive/UAH_Project/datasets/raw/UAH-DRIVESET-v1/D6/20151221120051-26km-D6-AGGRESSIVE-MOTORWAY
/content/drive/MyDrive/UAH_Project/datasets/raw/UAH-DRIVESET-v1/uah_driveset_reader/icons


In [66]:
import importlib
import data_loader

importlib.reload(data_loader)

trip_paths = data_loader.get_all_trip_paths(DATASET_PATH)

print(len(trip_paths))

for trip in trip_paths[-5:]:
    print(trip)

40
/content/drive/MyDrive/UAH_Project/datasets/raw/UAH-DRIVESET-v1/D6/20151217162714-26km-D6-NORMAL-MOTORWAY
/content/drive/MyDrive/UAH_Project/datasets/raw/UAH-DRIVESET-v1/D6/20151217164730-25km-D6-DROWSY-MOTORWAY
/content/drive/MyDrive/UAH_Project/datasets/raw/UAH-DRIVESET-v1/D6/20151221112434-17km-D6-NORMAL-SECONDARY
/content/drive/MyDrive/UAH_Project/datasets/raw/UAH-DRIVESET-v1/D6/20151221113846-16km-D6-DROWSY-SECONDARY
/content/drive/MyDrive/UAH_Project/datasets/raw/UAH-DRIVESET-v1/D6/20151221120051-26km-D6-AGGRESSIVE-MOTORWAY


In [67]:
X, y, groups = build_dataset(DATASET_PATH)

print(X.shape)
print(y.shape)
print(groups.shape)

print(len(np.unique(groups)))

(30676, 120, 13)
(30676,)
(30676,)
40


In [69]:
import importlib
import lstm_model

importlib.reload(lstm_model)

<module 'lstm_model' from '/content/drive/MyDrive/UAH_Project/src/lstm_model.py'>

In [71]:
from pathlib import Path

lstm_path = PROJECT_PATH / "src" / "lstm_model.py"

print(lstm_path.read_text())

"""
lstm_model.py

Baseline LSTM model for driver behaviour classification.
"""

import torch
import torch.nn as nn


class LSTMClassifier(nn.Module):

    def __init__(
        self,
        input_size=13,
        hidden_size=64,
        num_layers=2,
        dropout=0.3,
        num_classes=3,
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout,
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 32),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(32, num_classes),
        )

    def forward(self, x):

        _, (hidden, _) = self.lstm(x)

        last_hidden = hidden[-1]

        logits = self.classifier(last_hidden)

        return logits


In [72]:
import lstm_model

print(dir(lstm_model))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__']


In [74]:
import traceback

try:
    import importlib
    import lstm_model
    importlib.reload(lstm_model)
except Exception:
    traceback.print_exc()

In [75]:
from lstm_model import LSTMClassifier

model = LSTMClassifier()

print(model)

LSTMClassifier(
  (lstm): LSTM(13, 64, num_layers=2, batch_first=True, dropout=0.3)
  (classifier): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=3, bias=True)
  )
)


In [76]:
import torch

dummy = torch.randn(32, 120, 13)

output = model(dummy)

print("Input Shape :", dummy.shape)
print("Output Shape:", output.shape)

print(output[:5])

Input Shape : torch.Size([32, 120, 13])
Output Shape: torch.Size([32, 3])
tensor([[-0.0646, -0.0801, -0.1726],
        [-0.0369, -0.0759, -0.1180],
        [-0.0800, -0.0680, -0.1572],
        [-0.0636, -0.0794, -0.1393],
        [-0.1052, -0.1086, -0.1633]], grad_fn=<SliceBackward0>)
